<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/3_1_Classification_and_Regression_Quality_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part III. Evaluation Toolkit and Model Selection. 8. Classification and Regression Quality Metrics

### 7.1. Матрица ошибок и базовые метрики бинарной классификации

Прежде чем обсуждать, как выбирать и валидировать модели, необходимо понять, *что именно* мы оптимизируем и сравниваем. Интуитивно понятная accuracy часто оказывается обманчивой. Представьте задачу обнаружения редкой болезни: модель, всегда предсказывающая «здоров», даст accuracy 99,9 % — но будет совершенно бесполезна. Эта глава даёт систематический инструментарий метрик для задач классификации и регрессии, чтобы вы могли осознанно выбирать критерий качества под свою прикладную задачу.

---

#### 1. Матрица ошибок (Confusion Matrix)

В бинарной классификации каждый объект обладает истинной меткой (положительный класс, 1, или отрицательный класс, 0) и получает предсказание модели (1 или 0). Сравнивая истину и предсказание, мы относим объект к одному из четырёх исходов (табл. 1). Геометрически множество всех объектов размера \(N\) разбивается на четыре непересекающиеся ячейки — это и есть **матрица ошибок** размером \(2 \times 2\).

**Таблица 1. Матрица ошибок бинарной классификации**

|                     | Предсказан положительный (1) | Предсказан отрицательный (0) |
|---------------------|-------------------------------|-------------------------------|
| **Истинный положительный (1)** | True Positive (TP)            | False Negative (FN)           |
| **Истинный отрицательный (0)** | False Positive (FP)           | True Negative (TN)            |

**Производные величины** выражаются через элементы матрицы:

- **TPR** (True Positive Rate, sensitivity, recall, полнота)  
  \[
  \text{TPR} = \frac{TP}{TP + FN}
  \]
  — доля положительных объектов, правильно определённых моделью. Характеризует способность обнаруживать положительный класс.

- **TNR** (True Negative Rate, specificity)  
  \[
  \text{TNR} = \frac{TN}{TN + FP}
  \]
  — доля отрицательных объектов, правильно определённых моделью.

- **FPR** (False Positive Rate)  
  \[
  \text{FPR} = \frac{FP}{TN + FP} = 1 - \text{TNR}
  \]
  — вероятность ложной тревоги, то есть доля отрицательных объектов, ошибочно объявленных положительными.

- **FNR** (False Negative Rate)  
  \[
  \text{FNR} = \frac{FN}{TP + FN} = 1 - \text{TPR}
  \]
  — вероятность пропуска цели, доля положительных объектов, не обнаруженных моделью.

- **Precision** (точность)  
  \[
  \text{Precision} = \frac{TP}{TP + FP}
  \]
  — доля правильных положительных предсказаний среди всех объектов, которые модель назвала положительными. Отвечает на вопрос: «Насколько можно доверять положительному предсказанию?»

- **Accuracy** (общая доля правильных ответов)  
  \[
  \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
  \]

**Ключевой пример дисбаланса.** Рассмотрим датасет, в котором 95 % объектов принадлежат классу 0 и лишь 5 % — классу 1. Модель, всегда предсказывающая класс 0, имеет:

\[
TP = 0,\quad TN = 0.95N,\quad FP = 0,\quad FN = 0.05N,
\]
\[
\text{Accuracy} = \frac{0 + 0.95N}{N} = 0.95,\quad \text{TPR} = \frac{0}{0.05N} = 0.
\]

Accuracy равна 95 %, создавая иллюзию отличного качества, тогда как положительный класс полностью проигнорирован (TPR = 0). Этот пример показывает, что accuracy неприменима при сильном дисбалансе классов.

---

#### 2. Precision, Recall и компромисс между ними

Precision и Recall характеризуют качество предсказаний с разных сторон. Precision задаёт вопрос «Насколько мы уверены в положительном предсказании?», а Recall — «Какую долю всех положительных объектов мы обнаружили?». Между ними существует **обратная зависимость**: увеличивая Recall за счёт снижения порога классификации (модель охотнее предсказывает положительный класс), мы почти всегда уменьшаем Precision, и наоборот.

**Пример.** В задаче фильтрации спама высокий Recall важен, чтобы не пропустить ни одного спам-письма; высокий Precision — чтобы не отправить важное письмо в папку «Спам». Инженер выбирает компромисс, исходя из цены ошибок FP и FN.

Для формализации компромисса применяют **\(F_\beta\)-меру** — взвешенное гармоническое среднее Precision и Recall:

\[
F_\beta = (1 + \beta^2) \frac{\text{Precision} \cdot \text{Recall}}{\beta^2\,\text{Precision} + \text{Recall}}.
\]

Параметр \(\beta\) управляет относительной важностью Recall:

- \(\beta = 1\): равный вес Precision и Recall — классическая \(F_1\)-мера  
  \[
  F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}};
  \]
- \(\beta > 1\): Recall важнее Precision (например, \(\beta = 2\) даёт вдвое больший вес Recall);
- \(0 < \beta < 1\): Precision важнее Recall.

**Свойства \(F_\beta\)**:

- ограничена: \(0 \le F_\beta \le 1\);
- равна нулю, если Precision = 0 или Recall = 0;
- не зависит от TN — важное свойство при дисбалансе классов;
- гармоническое среднее штрафует за сильный разрыв между компонентами сильнее, чем арифметическое. Например, при Precision = 0,9 и Recall = 0,1 арифметическое среднее равно 0,5, тогда как гармоническое:  
  \[
  \frac{2 \cdot 0.9 \cdot 0.1}{0.9 + 0.1} = 0.18,
  \]
  что гораздо реалистичнее отражает плохое качество модели, «завалившей» одну из характеристик.

---

**Углублённый взгляд: вывод формулы \(F_\beta\) как взвешенного гармонического среднего**

Взвешенное гармоническое среднее величин \(x\) и \(y\) с положительными весами \(w_1, w_2\) определяется как
\[
H_w = \frac{w_1 + w_2}{\frac{w_1}{x} + \frac{w_2}{y}}
= \frac{1}{\frac{w_1}{w_1+w_2}\cdot\frac{1}{x} + \frac{w_2}{w_1+w_2}\cdot\frac{1}{y}}.
\]
Запишем \(F_\beta\) в эквивалентной форме:
\[
\frac{1}{F_\beta} = \frac{\beta^2\,\text{Precision} + \text{Recall}}{(1+\beta^2)\,\text{Precision}\,\text{Recall}}
= \frac{\beta^2}{1+\beta^2}\cdot\frac{1}{\text{Recall}} + \frac{1}{1+\beta^2}\cdot\frac{1}{\text{Precision}}.
\]
Следовательно,
\[
F_\beta = \frac{1}{\frac{\beta^2}{1+\beta^2}\cdot\frac{1}{\text{Recall}} + \frac{1}{1+\beta^2}\cdot\frac{1}{\text{Precision}}},
\]
что в точности совпадает с взвешенным гармоническим средним Precision и Recall, где вес Recall равен \(\beta^2\), а вес Precision равен \(1\). Нормировочный коэффициент \(1+\beta^2\) обеспечивает сумму весов, равную \(1+\beta^2\). При \(\beta=1\) оба веса равны \(1\) — получается обычное гармоническое среднее \(F_1\). При \(\beta=2\) вес Recall равен 4, а Precision — 1, то есть Recall в четыре раза важнее Precision.

---

#### 3. Многоклассовая классификация: микро- и макро-усреднение

При \(K\) классах матрица ошибок становится матрицей \(C\) размера \(K \times K\), где элемент \(C_{ij}\) — число объектов истинного класса \(i\), классифицированных как класс \(j\). Для каждого класса \(k\) можно определить собственные величины:

- \(TP_k = C_{kk}\) (диагональный элемент),
- \(FP_k = \sum_{j \neq k} C_{jk}\) (сумма столбца \(k\) без диагонального элемента),
- \(FN_k = \sum_{i \neq k} C_{ki}\) (сумма строки \(k\) без диагонального элемента).

Чтобы свести многоклассовую задачу к единой скалярной метрике, применяют усреднение.

**Микро-усреднение (micro-averaging)** сначала суммирует \(TP\), \(FP\) и \(FN\) по всем классам, а затем вычисляет единую метрику:

\[
TP_{\text{micro}} = \sum_{k=1}^{K} TP_k,\quad
FP_{\text{micro}} = \sum_{k=1}^{K} FP_k,\quad
FN_{\text{micro}} = \sum_{k=1}^{K} FN_k,
\]
\[
\text{Precision}_{\text{micro}} = \frac{TP_{\text{micro}}}{TP_{\text{micro}} + FP_{\text{micro}}},\quad
\text{Recall}_{\text{micro}} = \frac{TP_{\text{micro}}}{TP_{\text{micro}} + FN_{\text{micro}}}.
\]
Важное свойство: микро-усреднённая accuracy (\( (TP_{\text{micro}} + TN_{\text{micro}})/N \) с учётом всех классов) и микро-усреднённые Precision и Recall (при симметричном вычислении \(FN\) и \(FP\)) приводят к тому, что **микро-\(F_1\) совпадает с общей accuracy**, когда каждый объект имеет одну истинную метку и одно предсказание. Микро-метрики чувствительны к доминирующим классам: большой класс практически полностью определяет их значение.

**Макро-усреднение (macro-averaging)** вычисляет метрику для каждого класса независимо, а затем усредняет с равными весами:

\[
\text{Precision}_{\text{macro}} = \frac{1}{K}\sum_{k=1}^{K} \text{Precision}_k,\quad
\text{Recall}_{\text{macro}} = \frac{1}{K}\sum_{k=1}^{K} \text{Recall}_k,
\]
\[
F_{1,\text{macro}} = \frac{1}{K}\sum_{k=1}^{K} F_{1,k}.
\]
Макро-усреднение даёт равный вес каждому классу, независимо от его частоты. Оно особенно рекомендуется при сильном дисбалансе, когда важны все классы, в том числе минорные.

**Взвешенное макро-усреднение (weighted macro)** использует веса, пропорциональные поддержке (числу объектов) класса:
\[
\text{Precision}_{\text{weighted}} = \sum_{k=1}^{K} w_k\,\text{Precision}_k,\quad w_k = \frac{N_k}{N},
\]
что является компромиссом между микро- и макро-подходами.

**Наглядный пример.** Пусть три класса имеют поддержку \([1000, 10, 10]\) и матрицу ошибок, где модель хорошо распознаёт класс 1, но полностью игнорирует классы 2 и 3:  
\(TP_1 = 950\), \(FP_1 = 5\), \(FN_1 = 50\);  
\(TP_2 = 0\), \(FP_2 = 1\), \(FN_2 = 10\);  
\(TP_3 = 0\), \(FP_3 = 1\), \(FN_3 = 10\).  
Тогда:
\[
\text{Precision}_1 \approx 0.995,\; \text{Recall}_1 = 0.95,\; F_{1,1} \approx 0.972,
\]
\[
\text{Precision}_2 = 0,\; \text{Recall}_2 = 0,\; F_{1,2} = 0,
\]
\[
\text{Precision}_3 = 0,\; \text{Recall}_3 = 0,\; F_{1,3} = 0.
\]
Микро-\(F_1 \approx 0.96\), макро-\(F_1 \approx 0.324\). Микро-показатель создаёт впечатление почти идеальной модели, тогда как макро-\(F_1\) честно отражает провал на двух из трёх классов.

---

#### 4. Случайный классификатор и baseline

Для осмысленного сравнения метрик необходимо определить базовый уровень (baseline), достигаемый без обучения. В задаче со сбалансированными классами (50/50) случайное угадывание даёт accuracy ≈ 0.5. Для несбалансированного датасета (например, 90/10) наивный классификатор, всегда предсказывающий класс большинства, даёт accuracy = 0.9. Любая модель должна значимо превосходить этот baseline.

**Коэффициент Cohen’s Kappa** корректирует наблюдаемую точность на точность случайного классификатора, сохраняющего маргинальные частоты классов.

\[
\kappa = \frac{p_0 - p_e}{1 - p_e},
\]
где \(p_0\) — наблюдаемая accuracy, а \(p_e\) — ожидаемая точность при независимости предсказаний от истинных меток. Интерпретация: \(\kappa = 1\) — идеальное совпадение, \(\kappa = 0\) — уровень случайного угадывания, \(\kappa < 0\) — хуже случайного.

---

**Углублённый взгляд: вывод \(p_e\) через маргинальные вероятности**

Пусть \(p_{+}\) — истинная доля положительного класса, \(p^{*}\) — доля положительных предсказаний. Если предсказания модели и истинные метки независимы, то вероятность случайного совпадения положительных равна \(p_{+}p^{*}\), отрицательных — \((1-p_{+})(1-p^{*})\). Тогда
\[
p_e = p_{+}p^{*} + (1-p_{+})(1-p^{*}).
\]
В многоклассовом случае (K классов) аналогично:
\[
p_e = \sum_{k=1}^{K} p_k \, p^{*}_k,
\]
где \(p_k\) — истинная доля класса \(k\), \(p^{*}_k\) — доля предсказаний класса \(k\). Эта величина и подставляется в формулу \(\kappa\).

---

#### 5. Диагностические метрики: тестовая выборка и доверительные интервалы

Метрики, вычисленные на фиксированной тестовой выборке, дают лишь точечную оценку истинного качества модели. Для учёта неопределённости строят доверительные интервалы.

**Accuracy как биномиальная пропорция.** Правильность классификации каждого объекта можно рассматривать как испытание Бернулли с вероятностью успеха \(p\) (истинная accuracy). Точечная оценка \(\hat{p} = \text{Accuracy}\). Стандартная ошибка:
\[
SE = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}.
\]
Приближённый \((1-\alpha)\)-доверительный интервал Вальда:
\[
\hat{p} \pm z_{\alpha/2}\, SE,
\]
где \(z_{\alpha/2}\) — квантиль стандартного нормального распределения. Этот интервал неточен при малых выборках или \(\hat{p}\) вблизи 0 или 1.

**Интервал Вильсона** даёт более надёжное покрытие:
\[
\frac{\hat{p} + \frac{z^2}{2n} \pm z \sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}.
\]

**Бутстреп (bootstrap) для произвольных метрик.** Когда аналитическая формула стандартной ошибки сложна или неизвестна (например, для AUC), используют бутстреп. Из тестовой выборки размера \(n\) генерируют \(B\) выборок с повторением того же размера. Для каждой вычисляют целевую метрику \(\theta_b^*\). Эмпирическое распределение \(\{\theta_b^*\}\) позволяет получить:

- бутстреп-оценку стандартной ошибки \(\hat{SE}_{\text{boot}} = \sqrt{\frac{1}{B-1}\sum_{b=1}^B (\theta_b^* - \bar{\theta}^*)^2}\);
- \((1-\alpha)\)-доверительный интервал методом процентилей: \([\theta^*_{(\alpha/2)}, \theta^*_{(1-\alpha/2)}]\), где \(\theta^*_{(q)}\) — \(q\)-я выборочная квантиль бутстреп-распределения.

**Пример.** При оценке AUC модели на тестовой выборке из \(n\) объектов бутстреп позволяет получить не только среднюю AUC, но и её вариабельность, что критически важно при сравнении двух моделей.

---

### Вопросы для самопроверки

**Для начинающих**
1. Что показывают элементы матрицы ошибок TP, FP, FN и TN?
2. Почему accuracy может вводить в заблуждение на несбалансированных данных? Приведите пример.
3. В чём содержательное различие между Precision и Recall? Какие вопросы они задают о модели?
4. Что означает значение \(F_1 = 0.5\)? Может ли оно быть достигнуто при Precision = 0.9 и Recall = 0.1?
5. Как интерпретировать значение \(\kappa = 0.3\) по Cohen?
6. В какой ситуации предпочтительнее использовать макро-усреднение, а не микро-усреднение?

**Для профессионалов**
1. Выведите \(F_\beta\)-меру как взвешенное гармоническое среднее Precision и Recall с весами \(\beta^2\) и \(1\). Покажите, что при \(\beta=1\) получается обычное гармоническое среднее.
2. Докажите, что в многоклассовой классификации микро-\(F_1\) совпадает с общей accuracy при условии, что каждый объект имеет одну истинную метку.
3. Выведите формулу стандартной ошибки accuracy, рассматривая правильность ответа как испытание Бернулли. Какие предположения лежат в основе интервала Вальда?
4. Опишите процедуру построения доверительного интервала для AUC с помощью бутстрепа. Почему бутстреп предпочтительнее асимптотических формул?
5. Объясните, почему Cohen’s Kappa может быть чувствительна к дисбалансу классов, и как это влияет на её интерпретацию.

### 7.2. ROC-кривая, AUC и PR-кривая

Матрица ошибок и построенные на её основе метрики зависят от выбранного порога классификации. Однако во многих задачах важно оценить качество модели **в целом**, не привязываясь к конкретному порогу. Для этого используют кривые, показывающие поведение метрик при всех возможных порогах, и сводные скалярные характеристики — площадь под этими кривыми.

#### 1. ROC-кривая: определение и построение

Пусть классификатор для каждого объекта \(\mathbf{x}\) выдаёт числовую оценку \(s(\mathbf{x})\) (вероятность принадлежности к положительному классу, отступ или любое другое вещественное число). Правило решения вводится с помощью порога \(t\):

\[
\hat{y} =
\begin{cases}
1, & s(\mathbf{x}) \ge t, \\
0, & s(\mathbf{x}) < t.
\end{cases}
\]

При изменении порога \(t\) от \(-\infty\) до \(+\infty\) меняются величины \(TPR(t) = \frac{TP(t)}{TP(t)+FN(t)}\) и \(FPR(t) = \frac{FP(t)}{FP(t)+TN(t)}\). График зависимости \(TPR\) от \(FPR\) называется **ROC-кривой** (Receiver Operating Characteristic).

**Свойства ROC-кривой**:

- Диагональ \(TPR = FPR\) соответствует случайному классификатору, который не извлекает информации из признаков. Любая разумная модель должна лежать выше этой диагонали.
- Кривая всегда начинается в точке \((0,0)\) при \(t = +\infty\) (все объекты отнесены к отрицательному классу, нет ни одного ложного срабатывания, но и ни одного истинно положительного) и заканчивается в \((1,1)\) при \(t = -\infty\) (все объекты объявлены положительными, обнаружены все положительные, но и все отрицательные стали ложными срабатываниями).
- Идеальный классификатор проходит через точку \((0,1)\) — стопроцентный TPR при нулевом FPR. Чем ближе кривая к левому верхнему углу, тем лучше разделяющая способность модели.
- Выпуклость кривой вверх (вогнутость к точке \((0,1)\)) свидетельствует о хорошем ранжировании. Впадины (участки ниже диагонали) указывают на то, что модель в некоторых диапазонах работает хуже случайного угадывания; такие участки на практике можно исправить, инвертировав предсказания.

**Построение ROC-кривой по конечной выборке**. Пусть имеется \(n\) объектов с известными истинными метками \(y_i \in \{0,1\}\) и оценками \(s_i\). Алгоритм:
1. Сортируем объекты по убыванию \(s_i\).
2. Начинаем с порога \(t = +\infty\) (все предсказания отрицательные), начальная точка \((0,0)\).
3. Последовательно перемещаем порог вниз, при каждом шаге «активируя» следующий объект с наивысшей оценкой. Если объект действительно положительный (\(y=1\)), увеличиваем \(TP\), что повышает \(TPR\) (двигаемся вверх). Если отрицательный (\(y=0\)), увеличиваем \(FP\), что повышает \(FPR\) (двигаемся вправо).
4. Соединяя полученные точки ступенчатой линией, получаем эмпирическую ROC-кривую.

Геометрически это эквивалентно случайному блужданию на плоскости: каждый положительный объект добавляет шаг по вертикали высотой \(1/n_+\), каждый отрицательный — шаг по горизонтали длиной \(1/n_-\), где \(n_+\) и \(n_-\) — число положительных и отрицательных объектов соответственно.

#### 2. AUC (Area Under ROC Curve) — площадь под ROC-кривой

Естественной скалярной мерой качества, не зависящей от выбора порога, является **площадь под ROC-кривой**:

\[
\text{AUC} = \int_0^1 TPR(FPR) \, d(FPR).
\]

**Свойства AUC**:
- Для нетривиальных классификаторов \(0.5 \le \text{AUC} \le 1\). Если AUC < 0.5, это означает, что модель систематически путает классы — достаточно поменять знак оценок, чтобы получить AUC > 0.5.
- AUC = 1 соответствует идеальному разделению классов.
- AUC = 0.5 — случайному классификатору.
- В отличие от accuracy и \(F_1\), AUC **инвариантен к дисбалансу классов**. Он характеризует способность модели правильно упорядочивать объекты, не требуя калиброванных вероятностей.

**Вероятностная интерпретация**. Одно из важнейших свойств AUC состоит в том, что она равна вероятности того, что случайно выбранный положительный объект получит более высокую оценку, чем случайно выбранный отрицательный (с поправкой на равенство оценок):

\[
\text{AUC} = \mathbb{P}\bigl(s(\mathbf{x}_+) > s(\mathbf{x}_-)\bigr) + \frac{1}{2}\,\mathbb{P}\bigl(s(\mathbf{x}_+) = s(\mathbf{x}_-)\bigr).
\]

Эта интерпретация даёт прямой способ оценивания AUC без построения ROC-кривой и подчёркивает связь с качеством ранжирования.

---

**Углублённый взгляд: AUC как U-статистика Манна–Уитни**

Пусть \(n_+\) положительных объектов имеют оценки \(s_1^+, \dots, s_{n_+}^+\), а \(n_-\) отрицательных — \(s_1^-, \dots, s_{n_-}^-\). Определим статистику

\[
U = \frac{1}{n_+ n_-} \sum_{i=1}^{n_+} \sum_{j=1}^{n_-} \left[ \mathbf{1}[s_i^+ > s_j^-] + \frac{1}{2}\mathbf{1}[s_i^+ = s_j^-] \right].
\]

Покажем, что выборочная площадь под эмпирической ROC-кривой равна \(U\), что и устанавливает эквивалентность. При движении порога сверху вниз каждый положительный объект даёт вертикальный шаг, который добавляет к уже накопленному горизонтальному смещению, соответствующему числу отрицательных объектов с оценкой выше текущей. Формально, при сортировке всех \(n_+ + n_-\) объектов по убыванию оценок, площадь под ступенчатой кривой равна сумме по всем положительным объектам долей горизонтального отрезка, находящегося левее них. Если для данного положительного объекта \(k\) отрицательных имеют бо́льшую оценку, он вносит в площадь \(k / (n_+ n_-)\). Суммируя, получаем в точности \(U\).

Далее, \(U\) является несмещённой оценкой вероятности \(\mathbb{P}(s_+ > s_-) + \frac{1}{2}\mathbb{P}(s_+ = s_-)\), и как U-статистика она обладает асимптотической нормальностью. Это открывает путь к построению доверительных интервалов и статистических тестов.

---

#### 3. Статистические свойства AUC

Поскольку AUC вычисляется по конечной выборке, её точечная оценка обладает дисперсией, которую необходимо учитывать при сравнении моделей. Классический результат **Хэнли и Макнила (1982)** даёт формулу дисперсии AUC для одного классификатора:

\[
\text{Var}(\text{AUC}) = \frac{1}{n_+ n_-} \left[ \frac{\text{AUC}(1-\text{AUC}) + (n_+-1)(Q_1 - \text{AUC}^2) + (n_--1)(Q_2 - \text{AUC}^2)}{n_+ n_-} \right],
\]

где \(Q_1\) — вероятность того, что два случайно выбранных положительных объекта будут правильно ранжированы относительно одного случайного отрицательного, а \(Q_2\) — аналогичная вероятность для двух отрицательных и одного положительного. На практике дисперсию чаще оценивают с помощью бутстрепа, который не требует аналитических выражений и примени́м к любым метрикам.

Построение доверительного интервала для AUC может опираться на нормальное приближение \(\text{AUC} \pm z_{\alpha/2} \sqrt{\text{Var}(\text{AUC})}\) либо на бутстреп-процентили.

**Сравнение двух классификаторов по AUC**. Когда две модели оцениваются на одной и той же тестовой выборке, их AUC-оценки коррелированы. Тест **ДеЛонга (1988)** учитывает эту корреляцию и позволяет проверить гипотезу о равенстве истинных AUC. Он основан на многомерном обобщении U-статистики и является золотым стандартом в медицинской диагностике. В качестве непараметрической альтернативы можно применить бутстреп-тест, генерируя парные выборки и оценивая распределение разности AUC.

#### 4. Precision-Recall (PR) кривая

При сильном дисбалансе классов (например, 1 положительный на 1000 отрицательных) ROC-кривая может создавать излишне оптимистичное впечатление, поскольку огромное количество истинно отрицательных объектов (TN) «разбавляет» вклад ложных срабатываний в \(FPR\). В таких ситуациях более информативна **кривая точности-полноты (Precision-Recall, PR)**.

**Определение**. Для каждого порога \(t\) вычисляются \(\text{Recall}(t) = TPR(t)\) и \(\text{Precision}(t) = \frac{TP(t)}{TP(t) + FP(t)}\). Кривая строится как график \(\text{Precision}\) по оси ординат против \(\text{Recall}\) по оси абсцисс при изменении \(t\) от \(+\infty\) до \(-\infty\).

**Свойства**:
- Обычно кривая начинается в точке с очень высоким Precision и близким к нулю Recall (при высоком пороге), заканчиваясь в точке \((\text{Recall}=1,\, \text{Precision}=p)\), где \(p = \frac{n_+}{n_+ + n_-}\) — доля положительных объектов в выборке (при \(t = -\infty\) все объекты предсказаны положительными).
- Случайный классификатор даёт горизонтальную прямую \(\text{Precision} = p\).
- Идеальный классификатор проходит через точку \((1,1)\), что достижимо только при \(p=1\); в реалистичных задачах кривая не может достичь одновременно Recall = 1 и Precision = 1.

**Площадь под PR-кривой (PR-AUC, Average Precision, AP)** определяется как

\[
\text{AP} = \int_0^1 \text{Precision}(r) \, dr,
\]

где \(r\) — Recall. AP является средней точностью по всем уровням полноты и принимает значения от \(p\) (случайный классификатор) до 1 (идеальный). Эта метрика особенно чувствительна к ложным срабатываниям и потому более адекватна при сильном дисбалансе.

**Выбор между ROC и PR-AUC**:
- ROC-анализ предпочтителен, когда нужно сравнить общую разделяющую способность классификаторов, и дисбаланс не является экстремальным.
- PR-AUC следует использовать при сильном дисбалансе (1:1000 и более), когда важна именно ценность положительных предсказаний, а не глобальное ранжирование.

---

**Углублённый взгляд: связь ROC и PR-кривых**

Для фиксированного классификатора и заданного порога \(t\) выполняется соотношение:

\[
\text{Precision} = \frac{p \cdot \text{TPR}(t)}{p \cdot \text{TPR}(t) + (1-p) \cdot \text{FPR}(t)},
\]

где \(p\) — априорная доля положительного класса. Эта формула получается из определений: \(TP = p \cdot n \cdot TPR\), \(FP = (1-p) \cdot n \cdot FPR\), и подстановки в выражение \(\frac{TP}{TP+FP}\). Таким образом, каждая точка ROC-кривой однозначно отображается в точку PR-кривой при известном \(p\). Следовательно, PR-кривая может быть восстановлена из ROC-кривой, но сама PR-кривая зависит от баланса классов. Именно эта зависимость от \(p\) делает PR-AUC более «честной» мерой при сильном дисбалансе: две модели с одинаковой ROC-кривой дадут разные PR-кривые, если \(p\) различается. При сравнении моделей на одном и том же датасете это не проблема, но при переносе на другую популяцию с иной распространённостью класса PR-метрики изменятся, в то время как AUC (и ROC) останутся прежними — это подчёркивает важность фиксации контекста при интерпретации.

---

#### 5. Примеры и интерпретация

- **Идеальный классификатор**: существует порог, при котором все положительные объекты получают оценку выше, чем все отрицательные. ROC проходит через \((0,1)\), AUC = 1. PR-кривая проходит через \((1,1)\) (идеальная точность при любой полноте) — на практике она будет равна 1 везде, где Recall < 1, а при Recall = 1 может упасть до \(p\) (когда порог опускается настолько, что в положительные предсказания попадают отрицательные объекты). В случае \(n_+ \ll n_-\) идеал недостижим в точке \((1,1)\), но AP будет близка к 1.
- **Случайный классификатор**: ROC — диагональ, AUC = 0.5; PR — горизонтальная прямая на уровне \(p\), AP = \(p\).
- **Хорошо ранжирующий, но неточный классификатор при дисбалансе**. Рассмотрим выборку с \(n_+ = 10\), \(n_- = 990\) (\(p = 0.01\)). Классификатор выдаёт высокие оценки всем 10 положительным, низкие оценки 800 отрицательным и средние оценки оставшимся 190 отрицательным. Тогда:
  - При пороге, пропускающем только высокие оценки: \(TP = 10\), \(FP = 0\), \(TPR = 1\), \(FPR = 0\), \(\text{Precision} = 1\).
  - При пороге, захватывающем также средние оценки (чтобы обеспечить Recall = 1 при предсказании всех положительных, необходимо пропустить и те отрицательные, что имеют средние оценки, так как иначе какие-то положительные с наименьшими высокими оценками могут быть отсечены? В условии все положительные имеют высокие оценки, значит, порог «высокий» включает все положительные и ни одного отрицательного, и recall уже равен 1 при Precision = 1. Это противоречит замыслу примера. Скорректирую: пусть положительные объекты получили оценки, распределённые так: 10 положительных — оценки от 0.9 до 1.0; 800 отрицательных — от 0.0 до 0.1; 190 отрицательных — от 0.8 до 0.9. Тогда при пороге 0.9 мы захватываем только положительные: \(TP=10\), \(FP=0\), \(TPR=1\), \(FPR=0\), \(\text{Precision}=1\). Чтобы получить Recall < 1, можно взять порог > 0.9, но тогда не все положительные войдут. Лучше использовать классический пример: «все положительные имеют высокие оценки, часть отрицательных тоже имеют высокие оценки». Пусть все 10 положительных имеют оценку 0.95, а отрицательные: 800 с оценкой 0.1 и 190 с оценкой 0.9. Тогда при пороге 0.92 мы берём положительные (0.95) и отрицательные с 0.9? Нет, 0.9 < 0.92, так что они не войдут. Тогда все положительные и ни одного отрицательного — опять идеальная точка. Чтобы получить точку с recall=1 и Precision=0.05, нужно, чтобы recall=1 достигался только при пороге, который пропускает много FP. Значит, все положительные должны иметь оценки не выше некоторых отрицательных. Переформулирую пример в соответствии с типичным объяснением: пусть классификатор выдаёт оценки, и при этом для достижения полноты 1 приходится включить в положительные предсказания много отрицательных. В условии промпта: "Классификатор выдаёт высокие оценки для всех 10 положительных, низкие — для 800 отрицательных, средние — для 190. Посчитай TPR, FPR, Precision при разных порогах. ROC остаётся близкой к идеалу (FPR = 190/990 = 0.192, TPR = 1), PR-AUC будет значительно ниже 1, так как Precision = 10/(10+190) = 0.05 при пороге, где recall=1." Это означает, что порог, при котором recall=1, должен захватывать и те 190 отрицательных со средними оценками. Следовательно, у 10 положительных оценки не максимальные, а тоже средние или ниже? Если у положительных оценки выше, чем у тех 190 средних, то можно выбрать порог между ними, чтобы получить recall=1 без FP. Но тогда не получится recall=1 с FP=190. Значит, в этом примере все 10 положительных имеют оценки, которые либо такие же, как у 190 средних, либо ниже, но выше, чем у 800 низких. Допустим, оценки положительных = 0.5, у 800 отрицательных = 0.1, у 190 отрицательных = 0.5 или 0.6. Тогда при пороге, скажем, 0.4, захватываются все 10 положительных и 190 отрицательных (у которых оценки 0.5/0.6). Тогда TP=10, FP=190, FPR=190/990=0.192, TPR=1, Precision=10/200=0.05. При более высоком пороге, например 0.55, положительные с 0.5 отсекаются, recall падает. Таким образом, можно выбрать порог так, чтобы получить точки ROC (0.192, 1) и PR (Recall=1, Precision=0.05). Это иллюстрирует нечувствительность ROC к огромному числу FP (из-за большого количества TN), в то время как PR резко падает. Я опишу это в тексте, уточнив распределение оценок.)

- **Численный пример дисбаланса**: \(n_+ = 10, n_- = 990\) (\(p=0.01\)). Пусть классификатор присвоил всем 10 положительным объектам оценку \(0.5\), \(800\) отрицательным — оценку \(0.1\), и \(190\) отрицательным — оценку \(0.5\). Рассмотрим два порога:
  - \(t = 0.4\): предсказываются как положительные все объекты с \(s \ge 0.4\), то есть все положительные (10) и 190 отрицательных со средними оценками. Тогда \(TP = 10\), \(FP = 190\), \(FN = 0\), \(TN = 800\). Получаем \(TPR = 1\), \(FPR = 190/990 \approx 0.192\), \(\text{Precision} = 10/200 = 0.05\).
  - \(t = 0.6\): ни один объект не проходит (порог выше всех оценок) — тривиальная точка \((0,0)\).
  - \(t = 0.5\): здесь важен способ обработки равенств. Если считать, что при \(s \ge t\) положительный класс, то при \(t=0.5\) входят и положительные (10), и 190 отрицательных с оценкой 0.5 — та же точка, что при 0.4. Если строгое неравенство, можно взять \(t = 0.5+\varepsilon\), тогда не войдут ни те ни другие — точка (0,0). Таким образом, кривая будет иметь вид: старт из (0,0), затем сразу прыжок в точку (0.192, 1) при снижении порога до 0.5, и далее горизонтальный отрезок до (1,1), когда порог опускается до 0.1 и захватываются остальные отрицательные. ROC будет ломаной: (0,0) → (0.192, 1) → (1,1). Площадь под такой ROC близка к 1 (AUC ≈ 0.904). PR-кривая: точки (Recall, Precision): при пороге до включения средних — (0,1) или (0,0)? При пороге выше 0.5 никого нет, Recall=0, Precision не определена; часто кривую начинают с точки (0,1) интерполяцией. При пороге, пропускающем и положительные, и средние, получаем (1, 0.05). Площадь под PR-кривой будет чуть больше 0.05 (если учесть начальный участок с высоким Precision). Это наглядно демонстрирует, что ROC может вводить в заблуждение при сильном дисбалансе.

Приведённый пример чётко показывает, почему в задачах с экстремальным дисбалансом (поиск редких событий, медицинская диагностика редких заболеваний) следует опираться на PR-AUC, а не только на ROC-AUC.

---

### Вопросы для самопроверки

**Для начинающих**
1. Как по имеющейся выборке построить ROC-кривую, не прибегая к программированию? Опишите шаги.
2. Что означает точка (0,1) на ROC-плоскости? Может ли реальный классификатор достичь её на конечной выборке?
3. Сформулируйте вероятностную интерпретацию AUC. Почему она интуитивно связывает AUC с качеством ранжирования?
4. В чём преимущество PR-кривой перед ROC-кривой при сильном дисбалансе классов?
5. Дайте определение Average Precision (AP). Какие значения AP соответствуют случайному и идеальному классификаторам?

**Для профессионалов**
1. Докажите эквивалентность AUC и U-статистики Манна–Уитни, выражающей вероятность \( \mathbb{P}(s_+ > s_-) + \frac{1}{2}\mathbb{P}(s_+ = s_-) \). Выведите несмещённую оценку AUC через индикаторные функции.
2. Поясните происхождение формулы Хэнли–Макнила для дисперсии AUC. Почему при сравнении двух классификаторов на одних и тех же данных некорректно использовать обычный t-тест для независимых выборок?
3. В чём состоит суть теста ДеЛонга для сравнения коррелированных AUC? В каких случаях оправдано применение бутстрепа вместо теста ДеЛонга?
4. Выведите соотношение \(\text{Precision} = \frac{p \cdot \text{TPR}}{p \cdot \text{TPR} + (1-p) \cdot \text{FPR}}\). Покажите, как с его помощью PR-кривая может быть построена из ROC-кривой, и объясните, почему PR-AUC зависит от распространённости \(p\).
5. Приведите пример двух классификаторов с одинаковой ROC-кривой, но разными PR-кривыми при одном и том же \(p\). Возможно ли это? Обоснуйте.

### 7.3. Log-loss, Brier score и калибровка вероятностей

Метрики, рассмотренные ранее, оценивают качество бинарных решений или ранжирования. Однако во многих приложениях модель должна выдавать не просто метки, а **вероятности** принадлежности к классам — хорошо откалиброванные и информативные. В этом разделе мы вводим основные вероятностные метрики и обсуждаем, как добиться надёжных вероятностных оценок.

#### 1. Log-loss (кросс-энтропия)

Пусть модель для каждого объекта \(i\) предсказывает вероятность \(p_i = \mathbb{P}(y_i = 1 \mid \mathbf{x}_i)\). Тогда **логарифмическая функция потерь (log-loss, бинарная кросс-энтропия)** на выборке размера \(n\) определяется как

\[
\text{LogLoss} = -\frac{1}{n}\sum_{i=1}^{n} \Bigl[ y_i \log p_i + (1-y_i)\log(1-p_i) \Bigr].
\]

**Свойства**:
- Область значений: \(0 \le \text{LogLoss} < \infty\). Нижняя грань 0 никогда не достигается на конечной выборке, так как требует идеальной уверенности \(p_i = y_i\) (что формально недопустимо для вероятностей, не равных 0 или 1, но log-loss стремится к 0, если модель выставляет всё более близкие к истине вероятности). Если модель уверенно ошибается — предсказывает \(p_i = 0\) при \(y_i = 1\) или \(p_i = 1\) при \(y_i = 0\) — log-loss обращается в \(+\infty\).
- Минимизация log-loss эквивалентна максимизации логарифма правдоподобия биномиальной модели: \(\sum [y_i\log p_i + (1-y_i)\log(1-p_i)] = \log \prod p_i^{y_i} (1-p_i)^{1-y_i}\). Таким образом, log-loss напрямую связана с принципом максимального правдоподобия и является стандартной функцией потерь для логистической регрессии и нейронных сетей с сигмоидным выходом.
- Логарифмическая потеря принадлежит к классу **согласованных скоринговых правил (strictly proper scoring rules)**: математическое ожидание потери минимально тогда и только тогда, когда предсказанные вероятности совпадают с истинными.

---

**Углублённый взгляд: proper scoring rules и логарифмическая потеря**

Скоринговое правило \(S(p, y)\) называют **proper** (согласованным), если для любого истинного распределения \(y \sim \text{Bernoulli}(\theta)\) выполняется
\[
\mathbb{E}_{y\sim\theta}[S(\theta, y)] \le \mathbb{E}_{y\sim\theta}[S(p, y)] \quad \forall p \in [0,1].
\]
Оно называется **strictly proper**, если равенство достигается только при \(p = \theta\). Покажем, что log-loss \(S(p, y) = -[y\log p + (1-y)\log(1-p)]\) является strictly proper. Для фиксированного \(\theta\) имеем
\[
\mathbb{E}_{y}[S(p,y)] = -[\theta \log p + (1-\theta)\log(1-p)].
\]
Это выражение как функция от \(p\) достигает минимума при \(p = \theta\), что легко проверить, взяв производную по \(p\) и приравняв к нулю:
\[
\frac{\partial}{\partial p} \mathbb{E}[S] = -\frac{\theta}{p} + \frac{1-\theta}{1-p} = 0 \;\Longrightarrow\; p = \theta.
\]
Вторая производная положительна. Следовательно, log-loss — strictly proper.

Напротив, accuracy (как функция вероятностного предсказания) не является proper scoring rule. Пусть модель предсказывает вероятность \(p\), а решение принимается с порогом \(0.5\). Тогда ожидаемая accuracy есть \(\mathbb{P}(y=1, p\ge 0.5) + \mathbb{P}(y=0, p<0.5)\). Минимизация этого выражения не требует \(p = \theta\); модель может «выиграть», выдавая экстремальные вероятности, даже если они плохо откалиброваны. Поэтому для обучения вероятностных моделей accuracy не годится, а log-loss — да.

---

#### 2. Brier score

Альтернативой log-loss служит **Brier score** (среднеквадратичная ошибка вероятностей):

\[
\text{Brier} = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i)^2.
\]

**Свойства**:
- Значения лежат в отрезке \([0,1]\). Нуль достигается при идеальном детерминированном предсказании \(p_i = y_i\); максимальная ошибка 1 возникает, например, когда модель уверенно ошибается (\(p_i=1\) при \(y_i=0\) и наоборот для всех объектов).
- В отличие от log-loss, Brier score «мягче» реагирует на грубые ошибки: штраф квадратичный, а не логарифмический, поэтому конечен даже при \(p_i = 0\) или \(1\) с неверной меткой.
- Brier score также является strictly proper scoring rule. Его ожидание \(\mathbb{E}[(p - y)^2] = \theta(1-p)^2 + (1-\theta)p^2\) достигает минимума при \(p=\theta\).

**Разложение Brier score (разложение Мерфи)**. Важнейшее свойство Brier score — возможность разложить его на составляющие, отвечающие за **калибровку** и **разрешающую способность** (refinement). Пусть все предсказанные вероятности разбиты на \(K\) бинов \(B_1, \dots, B_K\) по близким значениям \(p\). Обозначим \(n_k\) — число объектов в бине \(k\), \(\bar{p}_k = \frac{1}{n_k}\sum_{i\in B_k} p_i\) — среднее предсказание в бине, \(\bar{y}_k = \frac{1}{n_k}\sum_{i\in B_k} y_i\) — истинная доля положительных. Тогда

\[
\text{Brier} = \underbrace{\frac{1}{n}\sum_{k=1}^{K} n_k (\bar{p}_k - \bar{y}_k)^2}_{\text{Calibration (калибровка)}} \;+\; \underbrace{\frac{1}{n}\sum_{k=1}^{K} n_k \bar{y}_k (1 - \bar{y}_k)}_{\text{Refinement (разрешение)}} \;-\; \underbrace{\frac{1}{n}\sum_{k=1}^{K} n_k (\bar{p}_k - p_i)^2 ???}_{\text{Нет, это разложение другое.}}
\]
(Поправка: классическое разложение Мерфи выглядит так: Brier = Reliability (калибровка) − Resolution + Uncertainty. Uncertainty — это \(\bar{y}(1-\bar{y})\), дисперсия истинных меток. Но я изложу разложение, которое даёт калибровку и refinement, как указано в промпте: «первое слагаемое — калибровка (насколько средние предсказания близки к истинным частотам), второе — refinement (разрешающая способность)». Корректное разложение для бинарного случая:

\[
\text{Brier} = \frac{1}{n}\sum_{k} n_k (\bar{p}_k - \bar{y}_k)^2 - \frac{1}{n}\sum_{k} n_k (\bar{y}_k - \bar{y})^2 + \bar{y}(1-\bar{y}),
\]
где \(\bar{y}\) — общая доля положительных. Здесь слагаемые: Reliability (калибровка), Resolution (разрешение, мера того, насколько бины разделяют классы) и Uncertainty (неопределённость, Brier score константного предсказания \(\bar{y}\)). Однако промпт просит: «первое слагаемое — калибровка, второе — refinement (разрешающая способность)». В некоторых упрощённых изложениях разложение записывают как сумму калибровки и refinement без uncertainty. Для ясности приведу формулу в виде: Brier = Calibration + Refinement, где Calibration = \(\frac{1}{n}\sum n_k (\bar{p}_k - \bar{y}_k)^2\), а Refinement = \(\frac{1}{n}\sum n_k \bar{y}_k (1-\bar{y}_k)\). Это разложение получается, если записать Brier как сумму по бинам внутригрупповой дисперсии ошибок и сгруппированной ошибки среднего, но проверим: для каждого бина \(\sum_{i\in B_k} (p_i - y_i)^2 = \sum (p_i - \bar{p}_k)^2 + n_k (\bar{p}_k - \bar{y}_k)^2 + \sum (y_i - \bar{y}_k)^2\). Усредняя, внутрибиновая дисперсия \(p_i\) даёт дополнительное слагаемое, а \(\sum (y_i - \bar{y}_k)^2 = n_k \bar{y}_k (1-\bar{y}_k)\). Тогда Brier = \(\frac{1}{n}\sum n_k (\bar{p}_k - \bar{y}_k)^2 + \frac{1}{n}\sum n_k \bar{y}_k(1-\bar{y}_k) + \frac{1}{n}\sum_{k}\sum_{i\in B_k}(p_i - \bar{p}_k)^2\). Последний член — дисперсия предсказаний внутри бинов, обычно мала при хорошем бинировании. В промпте разложение дано именно как калибровка + refinement. Я приведу это разложение с комментарием, что оно приближённое, либо сошлюсь на классическое разложение Мерфи, но адаптирую под запрос. Для строгости я приведу разложение Мерфи в каноническом виде:

\[
\text{Brier} = \underbrace{\frac{1}{n}\sum_{k} n_k (\bar{p}_k - \bar{y}_k)^2}_{\text{Reliability}} \;-\; \underbrace{\frac{1}{n}\sum_{k} n_k (\bar{y}_k - \bar{y})^2}_{\text{Resolution}} \;+\; \underbrace{\bar{y}(1-\bar{y})}_{\text{Uncertainty}}.
\]

Но в промпте хотят «первое слагаемое — калибровка, второе — refinement». Я переформулирую: Brier можно представить как сумму вклада калибровки (отклонение средних предсказаний от истинных частот) и вклада разрешения (насколько условные средние \(\bar{y}_k\) отличаются друг от друга). Я напишу разложение, близкое к указанному, но прокомментирую его смысл. Например: \(\text{Brier} = \frac{1}{n}\sum n_k (\bar{p}_k - \bar{y}_k)^2 + \frac{1}{n}\sum n_k \bar{y}_k(1-\bar{y}_k) - \frac{1}{n}\sum n_k (\bar{p}_k - p_i)^2\), но последнее обычно включают в калибровку. Чтобы избежать путаницы, я опишу разложение Мерфи и покажу, что первое слагаемое отвечает за калибровку, а (Resolution + Uncertainty) характеризуют refinement — способность разделять классы. Я укажу, что refinement = Uncertainty - Resolution. Таким образом, Brier = Calibration + Refinement, где Refinement = \(\bar{y}(1-\bar{y}) - \frac{1}{n}\sum n_k (\bar{y}_k - \bar{y})^2\). Это и есть классическое разложение. Приведу формулу и объясню.

**Разложение Мерфи**:

\[
\text{Brier} = \underbrace{\frac{1}{n}\sum_{k=1}^{K} n_k (\bar{p}_k - \bar{y}_k)^2}_{\text{Калибровка (Reliability)}} \;+\; \underbrace{\bar{y}(1-\bar{y})}_{\text{Неопределённость (Uncertainty)}} \;-\; \underbrace{\frac{1}{n}\sum_{k=1}^{K} n_k (\bar{y}_k - \bar{y})^2}_{\text{Разрешение (Resolution)}}.
\]

Здесь Uncertainty — постоянная для выборки составляющая, равная Brier’у для наивного прогноза \(\bar{y}\). Разность Uncertainty − Resolution называют **Refinement**; она тем меньше, чем лучше модель разделяет классы (больше Resolution). Таким образом, Brier score тем меньше, чем лучше калибровка (Reliability близка к нулю) и чем выше разрешающая способность модели.

---

#### 3. Калибровка вероятностей

Вероятностная модель называется **калиброванной**, если для любого порога \(t\) доля истинно положительных исходов среди объектов с предсказанной вероятностью \(\approx t\) действительно составляет \(\approx t\). Формально:

\[
\mathbb{P}(y=1 \mid p = t) = t.
\]

На практике калибровку оценивают с помощью **калибровочной кривой (reliability diagram)**: предсказанные вероятности группируют в бины, для каждого бина вычисляют среднюю предсказанную вероятность и наблюдаемую частоту положительных исходов. Идеально калиброванная модель даёт точки, лежащие на диагонали.

Многие популярные классификаторы не выдают хорошо откалиброванных вероятностей «из коробки». Например, метод опорных векторов (SVM) генерирует вещественные отступы, не лежащие в \([0,1]\), а деревья решений дают кусочно-постоянные вероятности с резкими границами. Для получения надёжных вероятностных оценок применяют **методы калибровки**, которые настраивают отображение выходов модели в калиброванные вероятности, минимизируя log-loss на отдельной валидационной выборке.

**Platt scaling** — параметрический метод, предложенный для SVM. Пусть модель выдаёт некоторую числовую оценку \(f = f(\mathbf{x})\) (например, расстояние до разделяющей гиперплоскости). Итоговая вероятность строится как сигмоида:

\[
\mathbb{P}(y=1 \mid f) = \frac{1}{1 + \exp(A f + B)},
\]

где параметры \(A\) и \(B\) подбираются путём минимизации log-loss на валидационной выборке (логистическая регрессия на одномерном признаке \(f\)). Platt scaling даёт хорошие результаты, когда исходные оценки приблизительно нормально распределены внутри каждого класса.

**Изотоническая регрессия** — непараметрический метод, который не накладывает предположений о форме зависимости. Она ищет неубывающую ступенчатую функцию \(g(f)\), отображающую оценки в вероятности, такую, что минимизируется \(\sum (g(f_i) - y_i)^2\) при ограничении монотонности. Изотоническая регрессия обычно превосходит Platt scaling при больших объёмах валидационной выборки, но более склонна к переобучению при малых данных.

---

**Углублённый взгляд: вывод Platt scaling из логистической модели**

Покажем, что сигмоидальное отображение возникает естественно, если условные распределения оценок \(f\) при фиксированных классах являются нормальными с равными дисперсиями. Пусть

\[
f \mid y=1 \sim \mathcal{N}(\mu_1, \sigma^2),\quad f \mid y=0 \sim \mathcal{N}(\mu_0, \sigma^2).
\]

Применим теорему Байеса для апостериорной вероятности:

\[
\mathbb{P}(y=1 \mid f) = \frac{\mathbb{P}(f \mid y=1)\,\mathbb{P}(y=1)}{\mathbb{P}(f \mid y=1)\,\mathbb{P}(y=1) + \mathbb{P}(f \mid y=0)\,\mathbb{P}(y=0)}.
\]

Положим \(\pi = \mathbb{P}(y=1)\). Подставляя плотности нормальных распределений, получаем

\[
\mathbb{P}(y=1 \mid f) = \frac{\pi \frac{1}{\sqrt{2\pi}\sigma} e^{-\frac{(f-\mu_1)^2}{2\sigma^2}}}{\pi \frac{1}{\sqrt{2\pi}\sigma} e^{-\frac{(f-\mu_1)^2}{2\sigma^2}} + (1-\pi) \frac{1}{\sqrt{2\pi}\sigma} e^{-\frac{(f-\mu_0)^2}{2\sigma^2}}}.
\]

Сокращая общие множители и вынося экспоненту с \(f\), преобразуем выражение к сигмоидному виду:

\[
\mathbb{P}(y=1 \mid f) = \frac{1}{1 + \frac{1-\pi}{\pi} \exp\!\left( \frac{(f-\mu_1)^2 - (f-\mu_0)^2}{2\sigma^2} \right)}.
\]

Вычислим разность квадратов:
\[
(f-\mu_1)^2 - (f-\mu_0)^2 = -2f(\mu_1 - \mu_0) + (\mu_1^2 - \mu_0^2) = -2f \Delta\mu + (\mu_1^2 - \mu_0^2),
\]
где \(\Delta\mu = \mu_1 - \mu_0\). Тогда показатель экспоненты становится:
\[
\frac{-2f \Delta\mu + (\mu_1^2 - \mu_0^2)}{2\sigma^2} = -\frac{\Delta\mu}{\sigma^2} f + \frac{\mu_1^2 - \mu_0^2}{2\sigma^2}.
\]
Таким образом,
\[
\frac{1-\pi}{\pi} \exp\!\left( \cdots \right) = \exp\!\left( -\frac{\Delta\mu}{\sigma^2} f + \frac{\mu_1^2 - \mu_0^2}{2\sigma^2} + \ln\frac{1-\pi}{\pi} \right).
\]
Введём обозначения
\[
A = -\frac{\Delta\mu}{\sigma^2},\qquad B = \frac{\mu_1^2 - \mu_0^2}{2\sigma^2} + \ln\frac{1-\pi}{\pi}.
\]
Тогда апостериорная вероятность принимает в точности вид Platt scaling:
\[
\mathbb{P}(y=1 \mid f) = \frac{1}{1 + \exp(A f + B)}.
\]

Если дисперсии классов не равны, апостериорная вероятность уже не является сигмоидой с линейным аргументом; в таком случае изотоническая регрессия может лучше улавливать истинную форму зависимости.

---

#### 4. Expected Calibration Error (ECE)

Для численной оценки калибровки вводят **Expected Calibration Error** — средневзвешенное абсолютное отклонение точности от средней уверенности по бинам. Разобьём предсказанные вероятности на \(M\) бинов (обычно с равной шириной). Для бина \(B_m\) обозначим \(|B_m|\) — число объектов, \(\text{acc}(B_m) = \frac{1}{|B_m|}\sum_{i\in B_m} \mathbf{1}[\hat{y}_i = y_i]\) при пороге 0.5 (или как доля положительных среди предсказанных, но точнее ECE использует accuracy или долю положительных в зависимости от контекста; в стандартном определении ECE для калибровки берут долю положительных \(\bar{y}_m\) и сравнивают со средней предсказанной вероятностью \(\bar{p}_m\) без привязки к порогу). В литературе ECE обычно определяется как

\[
\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n} \bigl| \bar{y}_m - \bar{p}_m \bigr|,
\]

где \(\bar{y}_m\) — истинная доля положительных в бине, \(\bar{p}_m\) — среднее предсказание. Другие варианты: \(\text{acc}(B_m)\) может означать accuracy, но для калибровочной задачи сравнивается именно \(\bar{y}_m\) (доля положительных) с \(\bar{p}_m\).

Свойства ECE:
- ECE измеряет только калибровку и игнорирует разрешающую способность. Две модели с одинаковым ECE могут иметь принципиально разный Brier score или log-loss.
- Модификация **Maximum Calibration Error (MCE)** берёт максимум по бинам вместо среднего, что важно в приложениях, критичных к наихудшему случаю калибровки.
- **Адаптивный ECE** разбивает бины так, чтобы в каждом было примерно одинаковое число объектов, обеспечивая более стабильную оценку.

---

#### 5. Практические рекомендации по выбору вероятностной метрики

- **Log-loss** — стандарт де-факто в соревнованиях и при обучении вероятностных моделей, поскольку строго штрафует за чрезмерно уверенные ошибочные предсказания. Рекомендован, когда важно не только ранжирование, но и точные вероятности, а также при наличии выбросов в оценках модель должна быть осторожной.
- **Brier score** удобен своей интерпретируемостью (всегда в \([0,1]\)) и меньшей чувствительностью к единичным экстремальным ошибкам. Он более робастен и хорошо подходит для сравнения моделей, когда log-loss может уходить в бесконечность из-за отдельных неверных предсказаний с вероятностью 0 или 1.
- **Когда калибровка критична**: в задачах с высокими ставками (медицинская диагностика, кредитный скоринг, страхование), где вероятности напрямую используются для принятия решений и расчёта ожидаемых потерь. Здесь следует обязательно анализировать калибровочные кривые и применять Platt scaling или изотоническую регрессию.
- **Когда калибровка не обязательна**: в задачах ранжирования (поиск, рекомендации), где достаточно правильного порядка объектов, и основными метриками являются AUC, MAP или precision@k. В таких сценариях монотонного преобразования оценок достаточно для сохранения качества, и дополнительная калибровка не улучшает бизнес-метрики.

---

### Вопросы для самопроверки

**Для начинающих**
1. Что измеряет log-loss и почему он резко возрастает, если модель предсказывает \(p=0\) для объекта с \(y=1\)?
2. Как интерпретировать значение Brier score, равное 0.2? Сравните с константным предсказанием доли положительного класса.
3. Что означает утверждение «модель хорошо откалибрована»? Как выглядит идеальная калибровочная кривая?
4. Для чего нужен Platt scaling и в каком виде он представляет вероятности?
5. Что отражает величина ECE? Приведите пример двух моделей с одинаковым log-loss, но разным ECE.

**Для профессионалов**
1. Докажите, что log-loss является strictly proper scoring rule, а accuracy — нет. Приведите пример, когда максимизация accuracy приводит к некалиброванным вероятностям.
2. Выпишите разложение Мерфи для Brier score и объясните смысл каждого слагаемого. Покажите, как разрешающая способность (resolution) связана с дискриминацией классов.
3. Выведите формулу Platt scaling из предположения о нормальности условных распределений оценок с равными дисперсиями. Что изменится, если дисперсии различны?
4. Сравните ECE и log-loss как меры качества вероятностного прогноза. Может ли модель с идеальной калибровкой (ECE ≈ 0) иметь высокий log-loss? Обоснуйте.
5. В каких ситуациях изотоническая регрессия предпочтительнее Platt scaling? Какие ограничения есть у изотонической регрессии при малом размере калибровочной выборки?

### 7.4. Метрики регрессии: MSE, MAE, MAPE, sMAPE, R²

До сих пор мы обсуждали метрики для задач классификации, где отклик принимает дискретные значения. В задачах регрессии целевая переменная \(y\) непрерывна, и необходимо измерять, насколько близки предсказания \(\hat{y} = f(\mathbf{x})\) к истинным значениям. В этом разделе мы систематически рассмотрим основные метрики регрессии, их математические свойства и интерпретацию.

---

#### 1. Постановка задачи регрессии

Пусть имеется выборка из \(n\) объектов \(\{\mathbf{x}_i, y_i\}_{i=1}^n\), где \(y_i \in \mathbb{R}\) — истинное значение, а \(\hat{y}_i = f(\mathbf{x}_i)\) — предсказание модели. Ошибка (остаток) на \(i\)-м объекте определяется как

\[
e_i = y_i - \hat{y}_i.
\]

Желательными свойствами метрики качества регрессии являются:
- **Интерпретируемость** — метрика должна легко пониматься в контексте предметной области.
- **Устойчивость к выбросам** — единичные аномальные наблюдения не должны катастрофически влиять на оценку.
- **Масштабная инвариантность** или наличие естественной привязки к масштабу данных.
- **Симметричность** — перестановка \(y\) и \(\hat{y}\) (в разумном смысле) не должна радикально менять значение метрики.
- **Дифференцируемость** — важна, если метрика непосредственно используется как функция потерь в градиентных методах оптимизации.

---

#### 2. Mean Squared Error (MSE) и RMSE

**Среднеквадратичная ошибка (MSE)** — наиболее распространённая метрика в регрессии:

\[
\text{MSE} = \frac{1}{n}\sum_{i=1}^n e_i^2 = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2.
\]

**Свойства**:
- Дифференцируема, что делает её удобной для градиентной оптимизации.
- Квадратичный штраф за ошибку приводит к тому, что большие отклонения вносят непропорционально большой вклад. Это делает MSE крайне чувствительной к выбросам.
- MSE тесно связана с принципом максимального правдоподобия: если ошибки независимы и распределены нормально \(y_i - \hat{y}_i \sim \mathcal{N}(0, \sigma^2)\), то минимизация MSE эквивалентна максимизации логарифма правдоподобия.
- Допускает фундаментальное **bias‑variance разложение**, лежащее в основе теории обобщающей способности.

**Корень из MSE (RMSE)** возвращает метрику в исходные единицы измерения \(y\):

\[
\text{RMSE} = \sqrt{\text{MSE}}.
\]

RMSE удобнее для интерпретации («средняя ошибка в тех же единицах, что и целевая переменная»), однако по-прежнему чувствительна к выбросам, так как квадрат находится под корнем.

---

**Углублённый взгляд: bias‑variance разложение MSE**

Рассмотрим теоретико-вероятностную постановку. Пусть истинная зависимость есть \(y = \mu(\mathbf{x}) + \varepsilon\), где \(\mathbb{E}[\varepsilon] = 0\), \(\text{Var}(\varepsilon) = \sigma^2\), а модель \(\hat{f}(\mathbf{x})\) обучена по случайной выборке и является случайной величиной. Тогда ожидаемая MSE на новой точке \(\mathbf{x}_0\) раскладывается в сумму:

\[
\mathbb{E}\bigl[(y_0 - \hat{f}(\mathbf{x}_0))^2\bigr] = \bigl(\mu(\mathbf{x}_0) - \mathbb{E}[\hat{f}(\mathbf{x}_0)]\bigr)^2 + \text{Var}\bigl(\hat{f}(\mathbf{x}_0)\bigr) + \sigma^2.
\]

Первое слагаемое — квадрат смещения (bias²), второе — дисперсия предсказаний модели, третье — неустранимый шум. Это разложение показывает, что MSE штрафует как за систематическую ошибку, так и за разброс предсказаний. На практике для детерминированной модели на фиксированной тестовой выборке аналогично получаем

\[
\mathbb{E}[(y - \hat{y})^2] = (\mu - \hat{y})^2 + \text{Var}(\varepsilon),
\]

что обосновывает использование MSE как оценки суммы квадрата смещения и дисперсии шума.

---

#### 3. Mean Absolute Error (MAE)

**Средняя абсолютная ошибка** определяется как

\[
\text{MAE} = \frac{1}{n}\sum_{i=1}^n |e_i| = \frac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|.
\]

**Свойства**:
- Линейный рост штрафа с увеличением ошибки делает MAE значительно более устойчивой к выбросам, чем MSE. Единичный выброс с ошибкой в 100 единиц вносит в MAE вклад 100, а в MSE — 10 000.
- Функция \(|e|\) не дифференцируема в нуле, однако при оптимизации можно использовать субградиент (например, \(\text{sign}(e)\)).
- MAE является оптимальной предсказательной метрикой, если шум имеет распределение Лапласа: \(y \sim \text{Laplace}(\hat{y}, b)\). В этом случае минимизация MAE эквивалентна оценке максимального правдоподобия.

**Сравнение MSE и MAE**:
- Если катастрофические ошибки недопустимы, MSE предпочтительнее, так как жёстко штрафует большие промахи.
- MAE даёт более реалистичное представление о «типичной» ошибке при наличии выбросов.
- Рост отношения \(\text{MSE} / \text{MAE}^2\) (или просто \(\text{MSE} / \text{MAE}\)) служит диагностикой: резкое увеличение указывает на появление выбросов или тяжёлых хвостов в распределении ошибок.
- **Медианная интерпретация:** минимизация MAE приводит к предсказанию, равному условной медиане распределения \(y \mid \mathbf{x}\), тогда как минимизация MSE даёт условное среднее. Это следует из того, что для случайной величины \(Y\) функция \(a \mapsto \mathbb{E}|Y - a|\) достигает минимума при \(a = \text{median}(Y)\).

---

**Углублённый взгляд: оптимальность MAE для лапласовского шума**

Пусть ошибки независимы и имеют распределение Лапласа с плотностью \(p(e) = \frac{1}{2b}\exp(-|e|/b)\). Логарифмическая функция правдоподобия для наблюдений \(y_i\) с предсказаниями \(\hat{y}_i\) (рассматриваемыми как параметры) равна

\[
\ell = -\sum_{i=1}^n \frac{|y_i - \hat{y}_i|}{b} - n \ln(2b).
\]

Максимизация правдоподобия эквивалентна минимизации \(\sum |y_i - \hat{y}_i|\), то есть MAE. Таким образом, MAE — согласованная с функцией правдоподобия метрика для лапласовского шума, и её оптимальным предсказанием является медиана. Напротив, MSE оптимальна для нормального шума и даёт среднее.

---

#### 4. Процентные метрики: MAPE и sMAPE

Чтобы получить масштабно-инвариантную меру, ошибки выражают в процентах относительно истинного значения.

**MAPE (Mean Absolute Percentage Error):**

\[
\text{MAPE} = \frac{100\%}{n} \sum_{i=1}^n \left| \frac{y_i - \hat{y}_i}{y_i} \right|.
\]

**Преимущества:** интерпретируемость («в среднем модель ошибается на \(x\%\)»), независимость от масштаба данных.

**Недостатки:**
- Не определена при \(y_i = 0\) и стремится к бесконечности, если какое-либо \(y_i\) близко к нулю.
- Асимметрична: перестановка \(y_i\) и \(\hat{y}_i\) даёт другой результат (штраф за завышенное предсказание может отличаться от штрафа за заниженное).
- Экстремально штрафует ошибки на малых истинных значениях, даже если абсолютная ошибка невелика.
- Не является согласованной функцией потерь в смысле принятия решений (см. Hyndman, 2006).

**sMAPE (symmetric MAPE):**

\[
\text{sMAPE} = \frac{100\%}{n} \sum_{i=1}^n \frac{|y_i - \hat{y}_i|}{(|y_i| + |\hat{y}_i|)/2}.
\]

**Свойства:**
- Симметрична: значение не изменится при замене \(y_i \leftrightarrow \hat{y}_i\).
- Определена, когда хотя бы одно из значений не равно нулю; в случае \(y_i = \hat{y}_i = 0\) обычно полагают вклад равным 0.
- Ограничена сверху 200% (достигается, когда одна из величин равна нулю, а другая отлична от нуля).

**Недостатки:**
- При очень малых \(|y_i|\) и \(|\hat{y}_i|\) знаменатель мал, что может приводить к нестабильности, даже если абсолютная ошибка ничтожна.
- sMAPE не является «симметричной» в статистическом смысле, так как её ожидаемое значение не минимизируется на одной и той же статистике распределения.

**Альтернативы:** в задачах прогнозирования временных рядов часто рекомендуют **MASE (Mean Absolute Scaled Error)**, которая масштабирует MAE на среднюю ошибку наивного прогноза (например, прогноза предыдущим значением) и лишена многих проблем MAPE/sMAPE.

---

**Углублённый взгляд: критика MAPE и sMAPE (Hyndman, 2006)**

Р. Хайндман и соавторы показали, что MAPE не является согласованной метрикой для выбора модели и сравнения прогнозов. В частности:
- Минимизация MAPE не приводит к оценке, соответствующей какой-либо разумной характеристике распределения (среднему, медиане или квантилю). Оптимальный прогноз по MAPE зависит от формы распределения и может смещаться.
- MAPE не удовлетворяет неравенству треугольника, поэтому её трудно интерпретировать как «расстояние».
- Асимметрия MAPE ведёт к тому, что прогнозы, систематически занижающие истинные значения, могут получать неоправданно низкий MAPE.

sMAPE исправляет только симметричность, но сохраняет нестабильность при малых значениях и всё ещё не имеет чёткой теоретико-вероятностной основы. Поэтому для научных и производственных систем прогнозирования Хайндман рекомендует масштабированные метрики вроде MASE, либо метрики, основанные на функциях правдоподобия (например, непрерывный ранжированный вероятностный счёт — CRPS). В учебной и прикладной литературе MAPE по-прежнему часто используется, но его ограничения необходимо осознавать.

---

#### 5. Коэффициент детерминации R²

**R² (коэффициент детерминации)** измеряет долю дисперсии зависимой переменной, объяснённой моделью:

\[
R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2},
\]

где \(\bar{y} = \frac{1}{n}\sum y_i\). Числитель — сумма квадратов остатков (RSS), знаменатель — полная сумма квадратов (TSS).

**Интерпретации:**
- \(R^2\) показывает, какую часть вариации \(y\) «улавливает» модель.
- Для линейной регрессии со свободным членом \(R^2\) равен квадрату коэффициента корреляции Пирсона между фактическими и предсказанными значениями: \(R^2 = \rho_{y,\hat{y}}^2\).
- \(R^2 = 0\) соответствует модели, предсказывающей константу \(\bar{y}\); если модель предсказывает хуже константы, \(R^2\) может быть отрицательным.

**Свойства:**
- \(R^2 \le 1\); равенство достигается при идеальном совпадении \(y_i = \hat{y}_i\) для всех \(i\).
- Монотонно возрастает при добавлении новых признаков (даже шумовых) в модель, что не позволяет использовать его для отбора моделей разной сложности.
- \(R^2\) не является мерой адекватности модели: низкое \(R^2\) может быть следствием высокой зашумлённости данных при правильной спецификации модели; высокое \(R^2\) может скрывать систематические ошибки (например, нелинейность). Поэтому анализ остатков обязателен.

**Скорректированный R² (adjusted \(R^2\))** устраняет эффект автоматического роста при добавлении параметров:

\[
\bar{R}^2 = 1 - \frac{\text{RSS} / (n - p - 1)}{\text{TSS} / (n - 1)} = 1 - (1 - R^2)\frac{n-1}{n-p-1},
\]

где \(p\) — количество признаков (без свободного члена). Дополнительные шумовые признаки уменьшают \(\bar{R}^2\), что позволяет сравнивать модели с разным числом переменных.

---

**Углублённый взгляд: вывод скорректированного R² через несмещённую оценку дисперсии**

Истинный (популяционный) коэффициент детерминации можно определить как

\[
\rho^2 = 1 - \frac{\sigma^2_\varepsilon}{\sigma^2_y},
\]

где \(\sigma^2_\varepsilon\) — дисперсия ошибок, а \(\sigma^2_y\) — дисперсия зависимой переменной. Несмещённая оценка \(\sigma^2_y\) есть \(s^2_y = \frac{\text{TSS}}{n-1}\). В предположениях классической линейной регрессии несмещённая оценка дисперсии ошибок есть \(s^2_\varepsilon = \frac{\text{RSS}}{n-p-1}\). Подставляя эти оценки в выражение для \(\rho^2\), получаем

\[
\hat{\rho}^2 = 1 - \frac{\text{RSS}/(n-p-1)}{\text{TSS}/(n-1)},
\]

что в точности совпадает с формулой скорректированного \(R^2\). Отсюда видно, что \(\bar{R}^2\) стремится давать более объективную оценку объясняющей способности модели на генеральной совокупности, штрафуя за избыточную сложность. При \(n \to \infty\) \(\bar{R}^2 \to R^2\), так как поправочный множитель \(\frac{n-1}{n-p-1} \to 1\).

---

### Вопросы для самопроверки

**Для начинающих**
1. Что измеряет MSE и почему она особенно чувствительна к выбросам?
2. Чем MAE принципиально отличается от MSE с точки зрения устойчивости к аномальным наблюдениям?
3. Почему MAPE не определена при нулевых истинных значениях и как ведёт себя при очень малых \(y_i\)?
4. Как интерпретировать значение \(R^2 = 0.8\)? Означает ли это, что модель «хорошая»?
5. В какой ситуации RMSE удобнее для отчёта, чем MSE?
6. Зачем нужен скорректированный \(R^2\) и как он штрафует за лишние признаки?

**Для профессионалов**
1. Выведите bias‑variance разложение MSE и покажите, что математическое ожидание MSE на тестовой точке равно сумме квадрата смещения, дисперсии модели и неустранимого шума.
2. Докажите, что минимизация MAE даёт медиану условного распределения, и покажите оптимальность MAE для лапласовского шума.
3. Проанализируйте поведение sMAPE при малых истинных значениях \(y_i\) и предсказаниях \(\hat{y}_i\). Может ли sMAPE быть обманчиво высокой, если ошибка мала в абсолютном выражении?
4. Выведите формулу скорректированного \(R^2\), используя несмещённые оценки дисперсии ошибок и дисперсии \(y\). Объясните, почему \(\bar{R}^2\) может уменьшаться при добавлении шумового признака.
5. Изложите основные пункты критики MAPE по Хайндману (2006). Почему MAPE не является согласованной метрикой для сравнения прогнозов и какие альтернативы предлагаются?

### 7.5. Метрики ранжирования: MAP, NDCG, MRR

Задачи классификации и регрессии подразумевают независимое предсказание для каждого объекта. Однако в поиске информации, рекомендательных системах и вопросно-ответных приложениях ключевым является **порядок** объектов в ранжированном списке. Метрики ранжирования оценивают, насколько хорошо модель ставит релевантные объекты выше нерелевантных или упорядочивает их по степени важности.

---

#### 1. Постановка задачи ранжирования

Для каждого запроса \(q\) (пользователь, поисковый запрос, контекст) имеется набор документов-кандидатов \(D_q = \{d_1, \dots, d_n\}\). Каждому документу присвоена оценка релевантности \(r(d) \in \mathbb{R}_{\ge 0}\): в простейшем случае бинарная (\(0\) — нерелевантен, \(1\) — релевантен), в более общем — градуированная (например, \(0\) — плохо, \(1\) — приемлемо, \(2\) — хорошо, \(3\) — отлично). Модель выдаёт числовой скор \(s(d)\) для каждого документа. Ранжирование строится сортировкой по убыванию \(s(d)\).

Цель метрик ранжирования — количественно оценить, насколько высокорелевантные документы концентрируются в верхней части ранжированного списка. В отличие от метрик классификации, здесь критична позиция, а не просто бинарное решение.

**Типы задач**:
- *Information retrieval* — поиск документов по запросу, где важен порядок выдачи.
- *Recommender systems* — персонализированная рекомендация товаров, фильмов, музыки.
- *Learning to rank* — обучение моделей, оптимизирующих непосредственно метрики ранжирования (pairwise, listwise подходы).

---

#### 2. Mean Average Precision (MAP)

Для запроса с бинарной релевантностью можно ввести **Precision@k** — долю релевантных документов среди первых \(k\) выдачи:

\[
\text{P@k} = \frac{\text{число релевантных документов в top-}k}{k}.
\]

Однако Precision@k не учитывает, сколько всего релевантных документов существует. **Average Precision (AP)** для одного запроса восполняет этот пробел. Имея общее число релевантных документов \(R\) (для данного запроса), AP вычисляется как среднее точности на позициях, где встречаются релевантные документы:

\[
\text{AP} = \frac{1}{R} \sum_{k=1}^{n} \text{P@k} \cdot \mathbf{1}[\text{документ на позиции } k \text{ релевантен}],
\]

или, эквивалентно,

\[
\text{AP} = \frac{1}{R} \sum_{k:\, \text{rel}(k)=1} \text{P@k},
\]

где \(\text{rel}(k) = 1\), если документ на позиции \(k\) релевантен, и \(0\) иначе. Если \(R=0\), AP полагают равной нулю.

**MAP (Mean Average Precision)** есть среднее AP по множеству запросов \(\mathcal{Q}\):

\[
\text{MAP} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \text{AP}_q.
\]

**Свойства MAP**:
- \(0 \le \text{MAP} \le 1\).
- Одновременно отражает и точность, и полноту: Precision усредняется по всем уровням полноты, достигаемым при движении сверху вниз по ранжированному списку.
- Не чувствительна к точным позициям релевантных документов внутри top-k — важен лишь факт их наличия на этих позициях. Например, два релевантных документа на позициях 1 и 10 дадут вклад \(\frac{1}{1} + \frac{2}{10}\) в сумму AP, тогда как при позициях 1 и 2 вклад был бы \(\frac{1}{1} + \frac{2}{2}\).
- Идеально подходит для бинарной релевантности.

---

**Углублённый взгляд: связь MAP с AUC для ранжирования**

Покажем, что для одного запроса AP эквивалентен площади под кривой Precision-Recall (AP — это Average Precision, уже определённая ранее как площадь под PR-кривой, но здесь идёт речь о том же понятии). Действительно, при движении порога по скору от самого высокого до самого низкого мы последовательно включаем документы. Precision@k вычисляется в моменты, когда мы добавляем релевантный документ. Среднее этих значений с весом \(1/R\) даёт площадь под ступенчатой PR-кривой. Таким образом, AP — это не что иное, как **PR-AUC** для данного запроса. Поскольку PR-кривая связана с ROC-кривой через распространённость \(p = R/n\), MAP является усреднением PR-AUC по запросам. Для бинарной релевантности можно также показать, что MAP пропорционален среднему AUC для каждого запроса, где в роли положительного класса выступают релевантные документы, но с поправкой на то, что нерелевантные документы рассматриваются как отрицательные только до достижения полноты.

Формально, AP = \(\frac{1}{R} \sum_{i=1}^R \frac{i}{r_i}\), где \(r_i\) — позиция \(i\)-го релевантного документа. Нетрудно видеть, что это совпадает с формулой для площади под PR-кривой. Эта связь объясняет популярность MAP как усреднённой по запросам метрики качества ранжирования.

---

#### 3. Normalized Discounted Cumulative Gain (NDCG)

Когда релевантность градуирована (\(r_i \in \{0,1,2,\dots,r_{\max}\}\)), простые Precision-ориентированные меры не отражают нюансы. **NDCG** предлагает нормализованную меру «накопленной дисконтированной полезности».

**Cumulative Gain (CG@k)** — сумма релевантностей первых \(k\) документов:

\[
\text{CG@k} = \sum_{i=1}^{k} r_i.
\]

CG не учитывает позицию, что нежелательно: релевантный документ на 10-й позиции даёт такой же вклад, как на 1-й.

**Discounted Cumulative Gain (DCG@k)** вводит логарифмический дисконт, понижающий вес документов с ростом позиции:

\[
\text{DCG@k} = \sum_{i=1}^{k} \frac{r_i}{\log_2(i+1)}.
\]

Часто используют экспоненциальную форму релевантности, подчёркивающую разницу между степенями:

\[
\text{DCG@k} = \sum_{i=1}^{k} \frac{2^{r_i} - 1}{\log_2(i+1)}.
\]

В этом варианте высокая релевантность (например, \(r=3\)) даёт вклад \(7 / \log_2(i+1)\), а низкая (\(r=1\)) — только \(1 / \log_2(i+1)\), что реалистичнее отражает предпочтения пользователя.

**Ideal DCG (IDCG@k)** — DCG@k для идеального ранжирования, где документы упорядочены по убыванию истинной релевантности. **NDCG@k** нормализует DCG, приводя значения в \([0,1]\):

\[
\text{NDCG@k} = \frac{\text{DCG@k}}{\text{IDCG@k}}.
\]

**Свойства NDCG**:
- \(0 \le \text{NDCG@k} \le 1\). Значение 1 достигается, когда верхние \(k\) позиций в точности соответствуют идеальному порядку.
- Нормализация позволяет усреднять NDCG по запросам с разным числом релевантных документов.
- Параметр \(k\) выбирают исходя из приложения: NDCG@10 типичен для веб-поиска, NDCG@5 — для рекомендаций на главном экране.
- Благодаря экспоненциальной форме, NDCG резко штрафует, если документы с высокой релевантностью оказываются далеко в хвосте.

---

**Углублённый взгляд: логарифмический дисконт и модель поведения пользователя**

Оправдание дисконта \(1 / \log_2(i+1)\) лежит в моделях взаимодействия пользователя с ранжированным списком. Предположим, что пользователь просматривает результаты сверху вниз и с вероятностью \(p\) переходит к следующему документу, а с вероятностью \(1-p\) прекращает просмотр. Вероятность того, что пользователь дойдёт до позиции \(i\), равна \(p^{i-1}\). Если априорное распределение вероятности продолжения \(p\) равномерно на \([0,1]\), то ожидаемая вероятность достижения позиции \(i\) составит

\[
\int_0^1 p^{i-1} \, dp = \frac{1}{i}.
\]

Таким образом, документ на позиции \(i\) будет учтён пользователем с весом, пропорциональным \(1/i\). Логарифмический дисконт \(1 / \log_2(i+1)\) близок к \(1/i\), но имеет более мягкое затухание, что эмпирически лучше соответствует поведению пользователей, которые могут «перескакивать» через несколько результатов. Дисконт \(1 / \log_2(i+1)\) также делает вклад первой позиции в точности равным релевантности (деление на \(\log_2(1+1)=1\)), что удобно.

Если модель предполагает экспоненциальное затухание с фиксированной вероятностью \(p\), то дисконт был бы геометрическим: \(p^{i-1}\). Однако логарифмическая форма стала стандартом благодаря своей простоте и неплохому согласию с эмпирическими данными о распределении кликов.

---

#### 4. Mean Reciprocal Rank (MRR)

В задачах, где пользователь ищет **единственный правильный ответ** (вопросно-ответные системы, поиск фактов, навигационные запросы), качество определяется тем, насколько быстро он найдёт первый релевантный документ.

**Reciprocal Rank (RR)** для одного запроса:

\[
\text{RR} = \frac{1}{\text{rank первого релевантного документа}}.
\]

Если релевантный документ отсутствует в выдаче, полагают \(\text{RR} = 0\).

**Mean Reciprocal Rank (MRR)** усредняет RR по запросам:

\[
\text{MRR} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \text{RR}_q.
\]

**Свойства**:
- Простота и наглядность: MRR = 0.5 означает, что первый релевантный результат в среднем находится на второй позиции.
- Игнорирует все релевантные документы после первого. Если в задаче важно вернуть несколько релевантных результатов, MRR даёт неполную картину.
- Особенно популярна в академических бенчмарках вопросно-ответных систем.

---

#### 5. Другие метрики ранжирования и рекомендаций

В зависимости от сценария применяют и другие меры.

- **Precision@k** — доля релевантных среди top-k; не зависит от общего числа релевантных. Подходит, когда количество показов строго ограничено (рекламные блоки, карусель товаров).
- **Recall@k** — доля всех релевантных документов запроса, попавших в top-k. Полезна, если нужно оценить покрытие.
- **R-Precision** — Precision@R, где \(R\) — общее число релевантных документов. Идеально сбалансирована по Precision и Recall и равна AP в точке отсечения \(R\).
- **Hit Rate (HR@k)** — бинарный индикатор: есть ли хотя бы один релевантный документ в top-k. Применяется в рекомендательных системах, где важен сам факт попадания релевантного объекта в зону внимания.

**Сравнение метрик и выбор**:
- *Бинарная релевантность, важен порядок всех релевантных* → MAP (или R-Precision).
- *Градуированная релевантность* → NDCG.
- *Только первый релевантный результат* → MRR.
- *Фиксированный размер выдачи, без учёта общей полноты* → Precision@k, Recall@k, HR.
- *Обучение ранжированию*: метрики часто недифференцируемы, поэтому используют суррогатные функции потерь, аппроксимирующие NDCG или MAP в listwise или pairwise подходах.

---

### Вопросы для самопроверки

**Для начинающих**
1. Что такое Average Precision (AP) для одного запроса? Как по набору ранжированных документов вычислить AP вручную?
2. Как определяется MAP и почему её усредняют по запросам, а не по документам?
3. Чем NDCG отличается от MAP? В каких случаях предпочтительнее NDCG?
4. Когда целесообразно применять Mean Reciprocal Rank (MRR) вместо MAP?
5. Как интерпретировать значение NDCG@10 = 0.85? Можно ли утверждать, что пользователь найдёт всё необходимое в первой десятке?

**Для профессионалов**
1. Выведите выражение для AP через сумму Precision@k в точках, где встречаются релевантные документы. Покажите, что AP численно равно площади под PR-кривой.
2. Докажите эквивалентность MAP усреднённому по запросам PR-AUC. Сравните MAP с усреднением ROC-AUC; объясните, почему при бинарной релевантности MAP может быть более информативной при сильной несбалансированности данных.
3. Сравните стандартный логарифмический дисконт в DCG (\(1/\log_2(i+1)\)) с экспоненциальным преобразованием релевантности (\(2^{r_i}-1\)). Как выбор основания логарифма и форма дисконта влияют на относительный вес позиций?
4. Как выбрать параметр \(k\) в метриках Precision@k и NDCG@k для веб-поиска, рекомендаций фильмов и ответа на вопрос? Предложите обоснование с точки зрения пользовательского поведения.
5. Покажите, что MRR равен математическому ожиданию величины, обратной рангу первого релевантного документа. В предположении, что релевантный документ всегда присутствует и его ранг распределён геометрически, вычислите MRR аналитически. Какое свойство MRR особенно чувствительно к «везучести» выборки запросов?